# EXP-11: TF-IDF Turbo — Enhanced Clinical NLP Pipeline

**Competition:** SPR 2026 — Mammography Report Classification  
**Task:** Classify mammography reports (Portuguese) into BI-RADS 0–6  
**Metric:** Macro F1  
**Approach:** Enhanced TF-IDF features + multi-section extraction + 5-model ensemble + CV-optimized thresholds  
**Runtime:** CPU only, <5 min

In [ ]:
# =============================================================================
# Cell 1: Imports and Data Loading
# =============================================================================
import os, re, hashlib, warnings
import numpy as np
import pandas as pd
from itertools import product

from sklearn.model_selection import GroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from sklearn.pipeline import FeatureUnion
from sklearn.metrics import f1_score
from scipy.sparse import hstack, csr_matrix

import lightgbm as lgb

warnings.filterwarnings('ignore')
np.random.seed(42)

TRAIN_PATH = '/kaggle/input/competitions/spr-2026-mammography-report-classification/train.csv'
TEST_PATH  = '/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

N_FOLDS = 4
N_CLASSES = 7

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")
print(f"Class distribution:\n{train['target'].value_counts().sort_index()}")

In [ ]:
# =============================================================================
# Cell 2: Text Preprocessing — Clean + Extract Sections
# =============================================================================

def normalize_text(s: str) -> str:
    """Basic normalization: lowercase, collapse whitespace."""
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{2,}", "\n", s)
    return s


def extract_achados(s: str) -> str:
    """Extract the 'achados' (findings) section from the report."""
    match = re.search(
        r'achados[:\s]*(.*?)(?:impress[aã]o|conclus[aã]o|an[aá]lise comparativa|opini[aã]o|$)',
        s, flags=re.DOTALL
    )
    return match.group(1).strip() if match else ""


def extract_conclusion(s: str) -> str:
    """Extract the 'impressao/conclusao' (conclusion) section."""
    match = re.search(
        r'(?:impress[aã]o|conclus[aã]o|opini[aã]o)[:\s]*(.*?)(?:an[aá]lise comparativa|recomenda|$)',
        s, flags=re.DOTALL
    )
    return match.group(1).strip() if match else ""


def clean_text(s: str) -> str:
    """Full cleaning: normalize numbers, add spacing around units."""
    s = normalize_text(s)
    # Separate units for better tokenization
    s = re.sub(r'\b(cm|mm)\b', ' \\1 ', s)
    # Normalize numbers
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s


def stable_hash(s: str) -> str:
    return hashlib.md5(str(s).encode("utf-8")).hexdigest()


# Apply preprocessing
for df in [train, test]:
    df['normalized']  = df['report'].apply(normalize_text)
    df['achados']     = df['normalized'].apply(extract_achados)
    df['conclusion']  = df['normalized'].apply(extract_conclusion)
    df['clean_full']  = df['report'].apply(clean_text)
    df['clean_achados']    = df['achados'].apply(clean_text)
    df['clean_conclusion'] = df['conclusion'].apply(clean_text)

train['group'] = train['report'].apply(stable_hash)

print("Text preprocessing done.")
print(f"Achados found: {(train['achados'] != '').sum()} / {len(train)}")
print(f"Conclusion found: {(train['conclusion'] != '').sum()} / {len(train)}")

In [ ]:
# =============================================================================
# Cell 3: Dense Feature Engineering
# =============================================================================

def extract_birads_number(text: str) -> int:
    """Extract BI-RADS category number if explicitly mentioned. Returns -1 if not found."""
    match = re.search(r'(?:bi-?rads|categoria)\s*:?\s*(\d)', text)
    if match:
        return int(match.group(1))
    return -1


def extract_dense_features(df: pd.DataFrame) -> csr_matrix:
    """Build a comprehensive set of hand-crafted clinical features."""
    feat = pd.DataFrame(index=df.index)
    text = df['normalized']  # already lowercased
    achados = df['achados']

    # --- Length / structure features ---
    feat['report_length']  = text.str.len()
    feat['word_count']     = text.str.split().str.len().fillna(0).astype(int)
    feat['line_count']     = text.str.count('\n') + 1
    feat['achados_length'] = achados.str.len()
    feat['conclusion_length'] = df['conclusion'].str.len()
    feat['num_findings']   = achados.str.count(r'[\n\-\*\u2022]').fillna(0).astype(int)

    # --- Clinical keyword features ---
    feat['has_measurement']   = text.str.contains(r'\b(?:cm|mm|medindo)\b', regex=True).astype(int)
    feat['has_spiculation']   = text.str.contains(r'espiculad', regex=True).astype(int)
    feat['has_distortion']    = text.str.contains(r'distor[cç][aã]o arquitetural', regex=True).astype(int)
    feat['has_biopsy']        = text.str.contains(r'bi[oó]psia|carcinoma|resultado de cine', regex=True).astype(int)
    feat['has_nodule']        = text.str.contains(r'n[oó]dulo|n[oó]dulos', regex=True).astype(int)
    feat['has_calcification'] = text.str.contains(r'calcifica[cç][aã]o|calcifica[cç][oõ]es|microcalcifica', regex=True).astype(int)
    feat['has_asymmetry']     = text.str.contains(r'assimetria', regex=True).astype(int)
    feat['has_lymphnode']     = text.str.contains(r'linfonodo|axilar', regex=True).astype(int)
    feat['has_suspicious']    = text.str.contains(r'suspeit[oa]|malign[oa]', regex=True).astype(int)
    feat['has_benign']        = text.str.contains(r'benign[oa]|cisto[s]?\b', regex=True).astype(int)

    # --- BI-RADS explicit mention ---
    feat['has_birads_explicit'] = text.str.contains(r'bi-?rads|categoria\s*\d', regex=True).astype(int)
    feat['birads_mentioned']    = text.apply(extract_birads_number)

    # --- Section presence ---
    feat['has_conclusion'] = text.str.contains(r'impress[aã]o|conclus[aã]o', regex=True).astype(int)
    feat['has_achados']    = text.str.contains(r'achados', regex=True).astype(int)

    # --- Laterality ---
    feat['has_mama_direita']  = text.str.contains(r'mama direita', regex=True).astype(int)
    feat['has_mama_esquerda'] = text.str.contains(r'mama esquerda', regex=True).astype(int)
    feat['has_bilateral']     = text.str.contains(r'bilateral', regex=True).astype(int)

    # --- Additional clinical signals ---
    feat['has_gross_calc']    = text.str.contains(r'calcifica[cç][aã]o grosseira|calcifica[cç][aã]o vascular', regex=True).astype(int)
    feat['has_nodulo_espic']  = text.str.contains(r'n[oó]dulo\s+espiculad', regex=True).astype(int)
    feat['has_heterogeneo']   = text.str.contains(r'heterog[eê]ne', regex=True).astype(int)
    feat['has_irregular']     = text.str.contains(r'irregular', regex=True).astype(int)

    return csr_matrix(feat.values.astype(np.float32))


X_train_dense = extract_dense_features(train)
X_test_dense  = extract_dense_features(test)

print(f"Dense features: {X_train_dense.shape[1]}")

In [ ]:
# =============================================================================
# Cell 4: TF-IDF Feature Building (multi-section)
# =============================================================================

# --- Full text TF-IDF ---
word_tfidf_full = TfidfVectorizer(
    ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True
)
char_tfidf_full = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(3, 6), min_df=3, max_df=0.95,
    sublinear_tf=True, max_features=100_000
)

# --- Achados section TF-IDF ---
word_tfidf_achados = TfidfVectorizer(
    ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True
)
char_tfidf_achados = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(3, 5), min_df=2, max_df=0.95,
    sublinear_tf=True, max_features=50_000
)

# --- Conclusion section TF-IDF ---
word_tfidf_concl = TfidfVectorizer(
    ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True
)

# Fit on train, transform both
print("Fitting TF-IDF on full text...")
Xtr_word_full = word_tfidf_full.fit_transform(train['clean_full'])
Xte_word_full = word_tfidf_full.transform(test['clean_full'])

Xtr_char_full = char_tfidf_full.fit_transform(train['clean_full'])
Xte_char_full = char_tfidf_full.transform(test['clean_full'])

print("Fitting TF-IDF on achados...")
Xtr_word_ach = word_tfidf_achados.fit_transform(train['clean_achados'])
Xte_word_ach = word_tfidf_achados.transform(test['clean_achados'])

Xtr_char_ach = char_tfidf_achados.fit_transform(train['clean_achados'])
Xte_char_ach = char_tfidf_achados.transform(test['clean_achados'])

print("Fitting TF-IDF on conclusion...")
Xtr_word_concl = word_tfidf_concl.fit_transform(train['clean_conclusion'])
Xte_word_concl = word_tfidf_concl.transform(test['clean_conclusion'])

# Combine all sparse features
X_train_sparse = hstack([
    Xtr_word_full, Xtr_char_full,
    Xtr_word_ach, Xtr_char_ach,
    Xtr_word_concl
]).tocsr()

X_test_sparse = hstack([
    Xte_word_full, Xte_char_full,
    Xte_word_ach, Xte_char_ach,
    Xte_word_concl
]).tocsr()

# Full feature matrix (sparse + dense)
X_train_full = hstack([X_train_sparse, X_train_dense]).tocsr()
X_test_full  = hstack([X_test_sparse, X_test_dense]).tocsr()

y = train['target'].astype(int).values
groups = train['group'].values

print(f"Sparse features: {X_train_sparse.shape[1]}")
print(f"Total features:  {X_train_full.shape[1]}")

In [ ]:
# =============================================================================
# Cell 5: Model Definitions + OOF Training (GroupKFold, 4 folds)
# =============================================================================

# Model configs: (name, constructor, uses_full_features)
# SVC and Ridge work best on sparse-only; LGB/LR use full features
MODEL_CONFIGS = [
    (
        'svc',
        lambda: CalibratedClassifierCV(
            LinearSVC(class_weight='balanced', random_state=42, max_iter=15000, C=1.0),
            cv=3, method='sigmoid'
        ),
        False  # sparse only
    ),
    (
        'ridge',
        lambda: RidgeClassifier(class_weight='balanced', alpha=1.0),
        False  # sparse only
    ),
    (
        'lr',
        lambda: LogisticRegression(
            class_weight='balanced', C=1.0, max_iter=5000,
            solver='lbfgs', multi_class='multinomial', random_state=42, n_jobs=-1
        ),
        True  # full features
    ),
    (
        'lgb1',
        lambda: lgb.LGBMClassifier(
            class_weight='balanced', n_estimators=500, learning_rate=0.03,
            max_depth=7, num_leaves=63, min_child_samples=20,
            subsample=0.8, colsample_bytree=0.6,
            random_state=42, n_jobs=-1, verbose=-1
        ),
        True
    ),
    (
        'lgb2',
        lambda: lgb.LGBMClassifier(
            class_weight='balanced', n_estimators=400, learning_rate=0.05,
            max_depth=5, num_leaves=31, min_child_samples=50,
            subsample=0.7, colsample_bytree=0.5,
            random_state=123, n_jobs=-1, verbose=-1
        ),
        True
    ),
]

gkf = GroupKFold(n_splits=N_FOLDS)

# Storage for OOF predictions (probabilities)
oof_probas = {}  # model_name -> (n_train, N_CLASSES)
test_probas = {}  # model_name -> (n_test, N_CLASSES)

for model_name, model_fn, use_full in MODEL_CONFIGS:
    print(f"\n{'='*60}")
    print(f"Training: {model_name} (features={'full' if use_full else 'sparse'})")
    print(f"{'='*60}")

    X_tr = X_train_full if use_full else X_train_sparse
    X_te = X_test_full if use_full else X_test_sparse

    oof = np.zeros((len(train), N_CLASSES), dtype=np.float64)
    test_preds_folds = np.zeros((len(test), N_CLASSES), dtype=np.float64)

    for fold_idx, (tr_idx, va_idx) in enumerate(gkf.split(X_tr, y, groups)):
        X_fold_tr, X_fold_va = X_tr[tr_idx], X_tr[va_idx]
        y_fold_tr, y_fold_va = y[tr_idx], y[va_idx]

        model = model_fn()
        model.fit(X_fold_tr, y_fold_tr)

        # Get probabilities
        if hasattr(model, 'predict_proba'):
            va_proba = model.predict_proba(X_fold_va)
            te_proba = model.predict_proba(X_te)
        else:
            # RidgeClassifier: use decision_function -> softmax
            va_dec = model.decision_function(X_fold_va)
            te_dec = model.decision_function(X_te)
            # Softmax conversion
            def softmax(x):
                e = np.exp(x - x.max(axis=1, keepdims=True))
                return e / e.sum(axis=1, keepdims=True)
            va_proba = softmax(va_dec)
            te_proba = softmax(te_dec)

        oof[va_idx] = va_proba
        test_preds_folds += te_proba / N_FOLDS

        fold_preds = np.argmax(va_proba, axis=1)
        fold_f1 = f1_score(y_fold_va, fold_preds, average='macro')
        print(f"  Fold {fold_idx}: macro-F1 = {fold_f1:.4f}")

    oof_preds = np.argmax(oof, axis=1)
    overall_f1 = f1_score(y, oof_preds, average='macro')
    print(f"  >> OOF macro-F1 = {overall_f1:.4f}")

    oof_probas[model_name] = oof
    test_probas[model_name] = test_preds_folds

print("\nAll models trained.")

In [ ]:
# =============================================================================
# Cell 6: Ensemble Weight Optimization + CV-Optimized Threshold Search
# =============================================================================

model_names = list(oof_probas.keys())
n_models = len(model_names)

# --- Step 1: Find optimal blend weights via grid search ---
print("Searching for optimal ensemble weights...")

best_f1 = 0
best_weights = None

# Generate weight combinations (step=0.05, must sum to 1)
# To keep it tractable with 5 models, we do a coarse search first
weight_steps = np.arange(0, 1.05, 0.05)

# Coarse search: sample random weight combinations
np.random.seed(42)
n_trials = 10000
for _ in range(n_trials):
    raw = np.random.dirichlet(np.ones(n_models))
    # Round to nearest 0.05
    w = np.round(raw / 0.05) * 0.05
    if w.sum() == 0:
        continue
    w = w / w.sum()  # renormalize

    blend = sum(w[i] * oof_probas[model_names[i]] for i in range(n_models))
    preds = np.argmax(blend, axis=1)
    score = f1_score(y, preds, average='macro')

    if score > best_f1:
        best_f1 = score
        best_weights = w.copy()

print(f"Best ensemble weights (macro-F1={best_f1:.4f}):")
for name, w in zip(model_names, best_weights):
    print(f"  {name}: {w:.2f}")

# Build blended OOF and test probabilities
oof_blend = sum(best_weights[i] * oof_probas[model_names[i]] for i in range(n_models))
test_blend = sum(best_weights[i] * test_probas[model_names[i]] for i in range(n_models))

# --- Step 2: CV-optimized threshold search for minority classes ---
# We search thresholds for classes 0, 1, 3, 4, 5, 6 (class 2 is majority)
# Apply in order: 6 first (most severe), then 5, 4, 3, 1, 0
print("\nSearching for optimal thresholds...")

threshold_classes = [6, 5, 4, 3, 1, 0]  # order of application (highest priority first)
threshold_range = np.arange(0.05, 0.51, 0.01)


def apply_thresholds(proba, thresholds_dict):
    """Apply class-specific thresholds in priority order."""
    preds = np.argmax(proba, axis=1)
    for cls in [6, 5, 4, 3, 1, 0]:
        if cls in thresholds_dict:
            thr = thresholds_dict[cls]
            # Only override if not already assigned to a higher-priority class
            # Classes with higher priority are processed first
            mask = proba[:, cls] > thr
            # Don't override if already set by a higher-priority class
            for higher_cls in [c for c in [6, 5, 4, 3, 1, 0] if c != cls]:
                if higher_cls in thresholds_dict and threshold_classes.index(higher_cls) < threshold_classes.index(cls):
                    mask = mask & (preds != higher_cls)
            preds[mask] = cls
    return preds


# Greedy sequential threshold optimization
best_thresholds = {}
current_preds = np.argmax(oof_blend, axis=1).copy()
baseline_f1 = f1_score(y, current_preds, average='macro')
print(f"Baseline OOF macro-F1 (no thresholds): {baseline_f1:.4f}")

for cls in threshold_classes:
    best_cls_f1 = baseline_f1
    best_cls_thr = None

    for thr in threshold_range:
        trial_thresholds = {**best_thresholds, cls: thr}
        trial_preds = apply_thresholds(oof_blend, trial_thresholds)
        trial_f1 = f1_score(y, trial_preds, average='macro')

        if trial_f1 > best_cls_f1:
            best_cls_f1 = trial_f1
            best_cls_thr = thr

    if best_cls_thr is not None:
        best_thresholds[cls] = best_cls_thr
        baseline_f1 = best_cls_f1
        print(f"  Class {cls}: threshold={best_cls_thr:.2f}, macro-F1={best_cls_f1:.4f}")
    else:
        print(f"  Class {cls}: no improvement from thresholds")

final_oof_preds = apply_thresholds(oof_blend, best_thresholds)
final_oof_f1 = f1_score(y, final_oof_preds, average='macro')
print(f"\nFinal OOF macro-F1 with thresholds: {final_oof_f1:.4f}")
print(f"Optimal thresholds: {best_thresholds}")

In [ ]:
# =============================================================================
# Cell 7: Final Predictions + Clinical Guardrails + Submission
# =============================================================================

# Apply ensemble + thresholds to test
test_preds = apply_thresholds(test_blend, best_thresholds)
test['target'] = test_preds


def apply_clinical_guardrails(row):
    """Post-processing rules based on clinical knowledge."""
    text = str(row['report']).lower()
    pred = int(row['target'])

    # --- Strong explicit BI-RADS mentions ---
    # If the report explicitly states a BI-RADS category, trust it
    birads_match = re.search(r'(?:bi-?rads|categoria)\s*:?\s*(\d)', text)
    if birads_match:
        mentioned = int(birads_match.group(1))
        if 0 <= mentioned <= 6:
            # Strong override for explicit mentions
            return mentioned

    # --- Pathology-confirmed malignancy ---
    if re.search(r'resultado de cine grau 3|carcinoma|neoplasia maligna', text):
        return 6

    # --- Highly suspicious morphology ---
    if re.search(r'n[oó]dulo\s+espiculad', text):
        if pred < 4:
            return 5

    if 'espiculad' in text and 'distor' in text and pred < 4:
        return 5

    # --- Benign calcifications should not be high BI-RADS ---
    if re.search(r'calcifica[cç][aã]o grosseira|calcifica[cç][aã]o vascular', text):
        if pred >= 4:
            # Only downgrade if there are no other suspicious findings
            if not re.search(r'espiculad|suspeit|malign|distor', text):
                return 2

    return pred


test['target'] = test.apply(apply_clinical_guardrails, axis=1)

# Generate submission
submission = test[['ID', 'target']].copy()
submission['target'] = submission['target'].astype(int)
submission.to_csv('submission.csv', index=False)

print("Submission saved!")
print(f"\nPrediction distribution:\n{submission['target'].value_counts().sort_index()}")
print(f"\nSubmission preview:")
print(submission)